In [8]:
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision import models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [20]:
DATASET_PATH = "dataset/all"

NUM_CLASSES = 4

BATCH_SIZE = 8

NUM_EPOCHS = 30

NUM_FOLDS = 5

random.seed(42)
torch.manual_seed(42)

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [22]:
# Аугментация

train_transform = transforms.Compose([

    transforms.RandomRotation(360),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.1,0.1),
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

val_transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [23]:
# Загружаем картинки

image_paths = []

labels = []

classes = ["1","2","5","10"]

for label, cls in enumerate(classes):

    class_dir = os.path.join(DATASET_PATH, cls)

    for file in os.listdir(class_dir):

        if file.lower().endswith((".jpg",".jpeg",".png")):

            image_paths.append(

                os.path.join(class_dir,file)

            )

            labels.append(label)

print("Всего изображений:",len(image_paths))

Всего изображений: 77


In [24]:
class CoinDataset(Dataset):

    def __init__(

        self,

        image_paths,

        labels,

        transform=None

    ):

        self.image_paths = image_paths

        self.labels = labels

        self.transform = transform

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self,index):

        image = Image.open(

            self.image_paths[index]

        ).convert("RGB")

        label = self.labels[index]

        if self.transform:

            image = self.transform(image)

        return image,label

In [25]:
skf = StratifiedKFold(

    n_splits=NUM_FOLDS,

    shuffle=True,

    random_state=42

)

In [26]:
# Начинаем цикл по фолдам

fold_results = []

best_accuracy = 0

for fold,(train_idx,val_idx) in enumerate(

        skf.split(

            image_paths,

            labels

        )):

    print()

    print("="*40)

    print(f"Fold {fold+1}")

    print("="*40)

    # формируем train/val

    train_images = [

        image_paths[i]

        for i in train_idx

    ]

    train_labels = [

        labels[i]

        for i in train_idx

    ]

    val_images = [

        image_paths[i]

        for i in val_idx

    ]

    val_labels = [

        labels[i]

        for i in val_idx

    ]

    # создаем набор данных

    train_dataset = CoinDataset(

        train_images,

        train_labels,

        train_transform

    )

    val_dataset = CoinDataset(

        val_images,

        val_labels,

        val_transform

    )


Fold 1

Fold 2

Fold 3

Fold 4

Fold 5


In [14]:
train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False

)

In [15]:
weights = models.MobileNet_V3_Small_Weights.DEFAULT

model = models.mobilenet_v3_small(

    weights=weights

)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\Users\Gwynbleidd/.cache\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:01<00:00, 6.13MB/s]


In [16]:
for param in model.features.parameters():

    param.requires_grad = False

In [19]:
model.classifier[3] = nn.Linear(

    model.classifier[3].in_features,

    NUM_CLASSES

)

model = model.to(device)